# ST-06 — Routage des EGE non résolus vers leur autorité d'enregistrement

Reprend les établissements dont la phase 3 se termine en `DOUTEUX` ou `REJETE`
et les oriente vers l'autorité compétente, qui arbitrera.

**Règle principale** : la catégorie d'établissement, croisée au département,
détermine l'autorité via le référentiel MOA.

**Rattrapage** : les EGE qui ressortent `A_ARBITRER` ou `NON_DETERMINEE` sont
repris avec l'autorité déclarée dans FINESS+, lue sur le dernier gestionnaire
ayant mis à jour la structure — `EGE.FK_MetadonneeId → Metadonnee → Gestionnaire`.
Une valeur territoriale ou `ANS` résout la structure ; sinon le classement
d'origine est conservé.

Le top 3 complet est conservé : le gestionnaire arbitre entre ces candidats.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd

from src import autorites as aut, routage as rt
from src.connexion   import get_finess_connection
from src.chargement  import resoudre_tables, charger_contacts
from src.display     import afficher_tableau
from config.settings import (
    ST_PHASE3, FINESS_EGE_CLEAN, REFERENTIEL_CATEGORIES, ROUTAGE_DIR,
)

CLE = rt.CLE_EGE

## 1. Référentiel des catégories

In [2]:
referentiel = aut.charger_referentiel(REFERENTIEL_CATEGORIES)
print(f"{len(referentiel):,} categories")
print(referentiel["statut_referentiel"].value_counts().to_string())
print()
print(referentiel["famille"].value_counts().to_string())

181 categories
statut_referentiel
RESOLVABLE         176
SANS_AUTORITE        3
MULTI_AUTORITES      2

famille
ARS            135
SOLIDARITES     46


## 2. Autorité déclarée dans FINESS+

In [3]:
conn = get_finess_connection()
TABLES = resoudre_tables(conn, {
    "pm": "PM_SMSSE", "ege": "EGE",
    "metadonnee": "Metadonnee", "gestionnaire": "Gestionnaire",
    "telecom": "Telecom",
    "ege_contact": "EgeContact",
})
ae_par_meta = rt.charger_ae_declaree(conn, TABLES)

print("\nCoordonnees :")
contacts = charger_contacts(conn, TABLES, "ege")
conn.close()

display(ae_par_meta[ae_par_meta["ae_declaree_finess"] != ""]["ae_declaree_finess"]
        .value_counts().head(20).to_frame("nb_metadonnees"))

   pm           -> T_FAC_PmSmsse
   ege          -> T_FAC_Ege
   metadonnee   -> T_FAC_Metadonnee
   gestionnaire -> T_FAC_Gestionnaire
   telecom      -> T_FAC_Telecom
   ege_contact  -> T_FAC_EgeContact
   158,254 metadonnees rattachees a un gestionnaire, 158,251 avec une autorite exploitable

Coordonnees :
   150,208 EGE avec au moins une coordonnee (86,562 mail, 145,805 telephone)


,nb_metadonnees
ae_declaree_finess,
ANS,58716
ARS-11,12127
ARS-75,9758
ARS-84,9712
ARS-76,8117
ARS-53,6156
ARS-93,6137
ARS-44,5887
ARS-32,5042


## 3. Structures à router

In [4]:
df = rt.charger_phase3(ST_PHASE3, col_cle=CLE,
                        col_statut="statut_structure")
print(f"{df[CLE].nunique():,} EGE, {len(df):,} lignes (top 3 conserve)")
print(df.drop_duplicates(CLE)["statut_structure"].value_counts().to_string())

7,648 EGE, 22,942 lignes (top 3 conserve)
statut_structure
DOUTEUX           6999
REJETE             602
NON_FIABLE_APE      47


## 4. Routage

In [5]:
resultats = rt.router_siretisation(df, referentiel, ae_par_meta)

rang1 = resultats[pd.to_numeric(resultats["rang"], errors="coerce") == 1]
print(rang1["autorite_statut"].value_counts().to_string())
print()
print("Origine de la resolution :")
print(rang1[rang1["autorite_statut"] == aut.RESOLU]["autorite_source"]
      .value_counts().to_string())
print()
print("Motifs des cas non resolus :")
print(rang1[rang1["autorite_statut"] != aut.RESOLU]["autorite_motif"]
      .value_counts().to_string())

   FK_MetadonneeId absente, rattrapage ignore
autorite_statut
RESOLU            7481
A_ARBITRER         158
NON_DETERMINEE       9

Origine de la resolution :
autorite_source
CATEGORIE    7481

Motifs des cas non resolus :
autorite_motif
categorie_multi_autorites     156
categorie_hors_referentiel      8
categorie_sans_autorite         2
departement_hors_perimetre      1


In [6]:
afficher_tableau(
    rang1.groupby(["autorite_code", "libelle_region"])[CLE].nunique()
         .reset_index(name="nb_egc").sort_values("nb_egc", ascending=False),
    "Repartition par autorite", max_lignes=25)

autorite_code,libelle_region,nb_egc
ARS-76,Occitanie,762
ARS-84,Auvergne-Rhône-Alpes,747
ARS-75,Nouvelle-Aquitaine,620
ARS-93,Provence-Alpes-Côte d'Azur,541
ARS-32,Hauts-de-France,446
ARS-28,Normandie,409
ARS-11,Île-de-France,388
ARS-27,Bourgogne-Franche-Comté,357
ARS-52,Pays de la Loire,323
ARS-53,Bretagne,321


## Coordonnées de contact

Lues directement dans FINESS+ : la liaison contact, puis la table Telecom
pivotée par canal. Les coordonnées de tous les contacts d'une structure sont
regroupées, ceux-ci étant souvent spécialisés — l'un porte le téléphone, l'autre
le mail.


In [7]:
resultats = rt.joindre_contacts(resultats, contacts, "NB_EgeId")


   TEL_MAIL         15,003 renseignes
   TEL_TELEPHONE    17,991 renseignes
   TEL_TELECOPIE    8,175 renseignes


## 6. Export

In [8]:
COLONNES_EXPORT = [
    "autorite_code", "libelle_autorite", "libelle_region", "departement",
    "autorite_statut", "autorite_source", "autorite_candidats", "autorite_motif",
    "ae_declaree_finess",
    "TX_NumFinessEge", "NB_EgeId", "NB_PmSmsseId", "TX_NomEgeLong",
    "TX_CategorieEntiteGeographiqueExercice", "libelle_categorie", "domaine_categorie",
    "adresse_complete_ege", "TEL_MAIL", "TEL_TELEPHONE",
    "TX_Siret", "siret_ref_app", "nom_etab_retenu",
    "adresse_complete_etab", "activitePrincipaleEtablissement",
    "score_nom", "score_adresse", "score_global", "score_global_approfondi",
    "statut_candidat", "statut_structure", "rang",
]
COLONNES_EXPORT = [c for c in COLONNES_EXPORT if c in resultats.columns]

recap = rt.exporter(resultats, ROUTAGE_DIR / "siretisation", "ST_P3",
                    col_cle=CLE, colonnes=COLONNES_EXPORT)
afficher_tableau(recap, "Classeurs ecrits", max_lignes=30)

attendu = resultats[CLE].nunique()
obtenu = int(recap["nb_structures"].sum())
assert attendu == obtenu, f"perte au decoupage : {attendu} != {obtenu}"
print(f"\n{obtenu:,} EGE repartis sur {len(recap)} classeurs")

autorite,region,fichier,nb_structures,nb_lignes,nb_departements
ARS-76,Occitanie,ST_P3_ARS-76_Occitanie.xlsx,762,2286,13
ARS-84,Auvergne-Rhône-Alpes,ST_P3_ARS-84_Auvergne-Rhone-Alpes.xlsx,747,2241,12
ARS-75,Nouvelle-Aquitaine,ST_P3_ARS-75_Nouvelle-Aquitaine.xlsx,620,1860,12
ARS-93,Provence-Alpes-Côte d'Azur,ST_P3_ARS-93_Provence-Alpes-Cote-d-Azur.xlsx,541,1623,6
ARS-32,Hauts-de-France,ST_P3_ARS-32_Hauts-de-France.xlsx,446,1338,5
ARS-28,Normandie,ST_P3_ARS-28_Normandie.xlsx,409,1227,5
ARS-11,Île-de-France,ST_P3_ARS-11_Ile-de-France.xlsx,388,1164,8
ARS-27,Bourgogne-Franche-Comté,ST_P3_ARS-27_Bourgogne-Franche-Comte.xlsx,357,1071,8
ARS-52,Pays de la Loire,ST_P3_ARS-52_Pays-de-la-Loire.xlsx,323,969,5
ARS-53,Bretagne,ST_P3_ARS-53_Bretagne.xlsx,321,963,4



7,648 EGE repartis sur 40 classeurs
